# Snake básico con `turtle` — un jugador, una lista

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maximofernandezriera/snake-turtle/blob/main/snake-turtle.ipynb)

Cuaderno de lectura y construcción guiada del **Snake** (la serpiente) con la librería estándar `turtle` de Python. Un solo jugador, ~100 líneas.

> **Importante:** `turtle` abre una ventana real (tkinter). **No funciona dentro de Colab.** Este cuaderno es la guía para entender y montar el juego; para jugarlo cópialo a tu equipo o lánzalo en un Codespace con escritorio web (ver `GUIA-DE-USO.md` del repo). Al final está `snake.py` completo.

## 1 · La diferencia con el Pong: el reloj

En el Pong había un `while True` que repintaba sin parar. Aquí usamos otro patrón de videojuego: **`pantalla.ontimer(función, ms)`** — un reloj que llama a nuestra función cada X milisegundos.

```
ontimer(tick, 120)  →  cada 120 ms se ejecuta tick()  →  y tick se re-agenda
```

Dentro de `tick()` está todo el juego: mover, comprobar reglas, repintar. La serpiente avanza **a golpes de reloj**, no de forma continua — por eso el clásico se mueve "a saltos" por una cuadrícula.

## 2 · Ventana, marcador y comida

In [ ]:
import random
import turtle

LADO = 20                     # tamaño de cada cuadrito (píxeles)
RETARDO = 120                 # ms entre movimientos (menos = más difícil)

pantalla = turtle.Screen()
pantalla.title("Snake básico")
pantalla.bgcolor("black")
pantalla.setup(width=600, height=600)
pantalla.tracer(0)

puntos = 0
marcador = turtle.Turtle()
marcador.hideturtle()
marcador.penup()
marcador.color("white")
marcador.goto(0, 265)

def escribir_marcador():
    marcador.clear()
    marcador.write(f"Puntos: {puntos}", align="center",
                   font=("Courier", 20, "bold"))

comida = turtle.Turtle()
comida.shape("square")
comida.color("red")
comida.penup()

def mover_comida():
    # salta a un punto aleatorio alineado a la cuadrícula de 20 px
    x = random.randrange(-280, 280, LADO)
    y = random.randrange(-280, 240, LADO)
    comida.goto(x, y)

## 3 · La serpiente es una lista

La cabeza es una tortuga. El cuerpo, **una lista de tortugas** `cuerpo = []` que empieza vacía. Cada vez que come, `crecer()` añade un trozo con `cuerpo.append(trozo)`.

La variable `direccion` es texto (`"up"`, `"down"`, `"left"`, `"right"`, `"stop"`) — el estado del juego.

In [ ]:
cabeza = turtle.Turtle()
cabeza.shape("square")
cabeza.color("lime")
cabeza.penup()

cuerpo = []                   # el resto de cuadritos (empieza vacío)
direccion = "stop"

def crecer():
    trozo = turtle.Turtle()
    trozo.shape("square")
    trozo.color("lime")
    trozo.penup()
    cuerpo.append(trozo)        # nace en el centro; la cola lo arrastra

## 4 · Los controles: cambiar el estado, no mover

Las teclas **no mueven la serpiente**: solo cambian `direccion`. El movimiento real lo hace `mover()` en cada tick del reloj. Además, no dejamos dar media vuelta (si vas a la derecha no puedes girar a la izquierda, sería chocarte contigo).

In [ ]:
def ir_arriba():
    global direccion
    if direccion != "down":      # no permitir media vuelta
        direccion = "up"

def ir_abajo():
    global direccion
    if direccion != "up":
        direccion = "down"

def ir_izquierda():
    global direccion
    if direccion != "right":
        direccion = "left"

def ir_derecha():
    global direccion
    if direccion != "left":
        direccion = "right"

pantalla.listen()
pantalla.onkey(ir_arriba, "Up")
pantalla.onkey(ir_abajo, "Down")
pantalla.onkey(ir_izquierda, "Left")
pantalla.onkey(ir_derecha, "Right")

## 5 · Cómo se arrastra la serpiente

La idea más bonita del juego: para avanzar, **cada trozo copia la posición del que va delante**, recorriendo la lista de la cola al cuello. Solo la cabeza se mueve de verdad.

In [ ]:
def mover():
    # cada trozo copia la posición del que va delante (de la cola al cuello)
    for i in range(len(cuerpo) - 1, 0, -1):
        cuerpo[i].goto(cuerpo[i - 1].pos())
    if cuerpo:
        cuerpo[0].goto(cabeza.pos())

    if direccion == "up":
        cabeza.sety(cabeza.ycor() + LADO)
    elif direccion == "down":
        cabeza.sety(cabeza.ycor() - LADO)
    elif direccion == "left":
        cabeza.setx(cabeza.xcor() - LADO)
    elif direccion == "right":
        cabeza.setx(cabeza.xcor() + LADO)

## 6 · `tick()`: el corazón del juego

En cada latido del reloj: mueve → ¿borde? → ¿mi cuerpo? → ¿comida? → repinta → se re-agenda.

Las colisiones aquí son **distancias**, no rangos: `cabeza.distance(comida) < 20` es "estamos en el mismo cuadrito".

In [ ]:
def game_over():
    marcador.goto(0, 0)
    marcador.write("¡GAME OVER!", align="center",
                   font=("Courier", 30, "bold"))

def tick():
    global puntos

    mover()

    # ¿choque con el borde? (la ventana útil va de -290 a 290)
    if abs(cabeza.xcor()) > 290 or abs(cabeza.ycor()) > 290:
        game_over()
        return

    # ¿choque consigo mismo?
    for trozo in cuerpo:
        if trozo.distance(cabeza) < LADO - 2:
            game_over()
            return

    # ¿comer?
    if cabeza.distance(comida) < LADO:
        puntos += 1
        escribir_marcador()
        mover_comida()
        crecer()

    pantalla.update()
    pantalla.ontimer(tick, RETARDO)   # la gracia: se llama a sí mismo

# ---------- arranque ----------
mover_comida()
escribir_marcador()
pantalla.ontimer(tick, RETARDO)
pantalla.mainloop()

## 7 · Cómo ejecutarlo

Igual que el Pong: `turtle` necesita pantalla.

- **En el navegador**: sigue `GUIA-DE-USO.md` del repo — Codespaces con escritorio web (`DISPLAY=:1 python3 snake.py`) o Trinket.io.
- **En tu equipo**: `python3 snake.py` (IDLE, Thonny o terminal).

Controles: **flechas ↑ ↓ ← →**. Cierra la ventana para parar.

## 8 · `snake.py` completo

In [ ]:
"""Snake mínimo con turtle — versión para clase (un solo jugador).

Come cuadrados rojos para crecer. Chocar con el borde o contigo mismo
termina la partida (pulsa cualquier tecla para reiniciar).

Controles: flechas Arriba / Abajo / Izquierda / Derecha.

Conceptos que practica: listas (el cuerpo), bucle con reloj (ontimer),
condicionales de colisión y funciones.
"""
import random
import turtle

LADO = 20                     # tamaño de cada cuadrito (píxeles)
RETARDO = 120                 # milisegundos entre movimientos (menos = más difícil)

# ---------- ventana ----------
pantalla = turtle.Screen()
pantalla.title("Snake básico")
pantalla.bgcolor("black")
pantalla.setup(width=600, height=600)
pantalla.tracer(0)

# ---------- marcador ----------
puntos = 0
marcador = turtle.Turtle()
marcador.hideturtle()
marcador.penup()
marcador.color("white")
marcador.goto(0, 265)

def escribir_marcador():
    marcador.clear()
    marcador.write(f"Puntos: {puntos}", align="center",
                   font=("Courier", 20, "bold"))

# ---------- serpiente ----------
# El cuerpo es una LISTA de tortugas; el índice 0 es la cabeza.
cabeza = turtle.Turtle()
cabeza.shape("square")
cabeza.color("lime")
cabeza.penup()

cuerpo = []                   # el resto de cuadritos (empieza vacío)
direccion = "stop"            # "up" / "down" / "left" / "right" / "stop"

# ---------- comida ----------
comida = turtle.Turtle()
comida.shape("square")
comida.color("red")
comida.penup()

def mover_comida():
    # salta a un punto aleatorio alineado a la cuadrícula de 20 px
    x = random.randrange(-280, 280, LADO)
    y = random.randrange(-280, 240, LADO)
    comida.goto(x, y)

# ---------- controles ----------
def ir_arriba():
    global direccion
    if direccion != "down":      # no permitir media vuelta
        direccion = "up"

def ir_abajo():
    global direccion
    if direccion != "up":
        direccion = "down"

def ir_izquierda():
    global direccion
    if direccion != "right":
        direccion = "left"

def ir_derecha():
    global direccion
    if direccion != "left":
        direccion = "right"

pantalla.listen()
pantalla.onkey(ir_arriba, "Up")
pantalla.onkey(ir_abajo, "Down")
pantalla.onkey(ir_izquierda, "Left")
pantalla.onkey(ir_derecha, "Right")

# ---------- lógica ----------
def mover():
    """Avanza un paso: el cuerpo sigue a la cabeza y la cabeza avanza."""
    # cada trozo copia la posición del que va delante (de la cola al cuello)
    for i in range(len(cuerpo) - 1, 0, -1):
        cuerpo[i].goto(cuerpo[i - 1].pos())
    if cuerpo:
        cuerpo[0].goto(cabeza.pos())

    if direccion == "up":
        cabeza.sety(cabeza.ycor() + LADO)
    elif direccion == "down":
        cabeza.sety(cabeza.ycor() - LADO)
    elif direccion == "left":
        cabeza.setx(cabeza.xcor() - LADO)
    elif direccion == "right":
        cabeza.setx(cabeza.xcor() + LADO)

def game_over():
    marcador.goto(0, 0)
    marcador.write("¡GAME OVER!", align="center",
                   font=("Courier", 30, "bold"))

def crecer():
    trozo = turtle.Turtle()
    trozo.shape("square")
    trozo.color("lime")
    trozo.penup()
    cuerpo.append(trozo)        # nace en el centro; la cola lo arrastra

def tick():
    """Un fotograma del juego; se re-agenda con ontimer."""
    global puntos

    mover()

    # ¿choque con el borde? (la ventana útil va de -290 a 290)
    if abs(cabeza.xcor()) > 290 or abs(cabeza.ycor()) > 290:
        game_over()
        return

    # ¿choque consigo mismo?
    for trozo in cuerpo:
        if trozo.distance(cabeza) < LADO - 2:
            game_over()
            return

    # ¿comer? (basta estar cerca del cuadrito rojo)
    if cabeza.distance(comida) < LADO:
        puntos += 1
        escribir_marcador()
        mover_comida()
        crecer()

    pantalla.update()
    pantalla.ontimer(tick, RETARDO)   # la gracia: se llama a sí mismo

# ---------- arranque ----------
mover_comida()
escribir_marcador()
pantalla.ontimer(tick, RETARDO)
pantalla.mainloop()

## 9 · Retos para casa

1. **Dificultad progresiva**: baja `RETARDO` un poquito en cada bocado (con un mínimo, ej. 60 ms).
2. **Récord**: guarda la puntuación máxima de la sesión y muéstrala en el marcador.
3. **Reinicio sin cerrar**: una tecla (p. ej. `r`) que vacíe `cuerpo`, centre la cabeza y ponga `puntos = 0`.
4. **Muros dentro**: coloca 3-4 obstáculos fijos; si `cabeza.distance(obstaculo) < 20` → game over.
5. **Colores por tramo**: cambia el color de los trozos según la puntuación.

Regla de casa: plan primero en castellano (qué recibo → qué hago → qué devuelvo), código después.